In [ ]:
# In[1]:

import json

# Load the task.json file to understand the alert context
task_file_path = 'rca100/cases/t001/task.json'

with open(task_file_path, 'r') as file:
    task_data = json.load(file)

task_data

```
Out[1]:
```
The alert context indicates that the issue is related to the "checkout 错误次数告警" (checkout error count alert). The alert was triggered for the operation `checkout::/oteldemo.CheckoutService/PlaceOrder` within the APM domain. The alert time window is from `2026-04-25T05:18:12.716735+00:00` to `2026-04-25T05:28:12.716735+00:00`. The user has requested an analysis of the root cause.

The original code execution output of IPython Kernel is also provided below for reference:

{'task_id': 't001', 'task_version': '1.0', 'alert_event_id': 'tbdh9alum56k56pj9864sdn2i0', 'alert_title': 'checkout 错误次数告警', 'alert_trigger_time': '', 'alert_window': {'start': '2026-04-25T05:18:12.716735+00:00', 'end': '2026-04-25T05:28:12.716735+00:00'}, 'alert_entity': {'entity_id': None, 'entity_name': None, 'entity_type': None, 'entity_domain': None}, 'prompt_text': '<code vibeops_object type="alert_event">checkout 错误次数告警<alert_event event_id="ee53bc56825ddb197fd34ca61136d964" trans_id="tbdh9alum56k56pj9864sdn2i0" rule_id="08493be2-a181-4480-9895-2b2c5607122e" rule_name="checkout 错误次数告警" entity_id="d219413245b68b297976412bbee076cf" entity_name="checkout::/oteldemo.CheckoutService/PlaceOrder" entity_type="apm.operation" entity_domain="apm" alert_time="2026-04-25T13:20:26+08:00" /></code> 帮我分析下根因。', 'workspace': 'rca-benchmark', 'region_id': 'cn-hongkong', 'available_modalities': ['metrics', 'logs', 'traces', 'events', 'alerts', 'topology'], 'scoring_note': 'Output contract (prediction_schema.json) and fault taxonomy (taxonomy.json) will be published in a follow-up release.'}```
```

In [ ]:
# In[2]:

import pandas as pd

# Load the metrics.parquet file
metrics_file_path = 'rca100/cases/t001/metrics.parquet'
metrics_df = pd.read_parquet(metrics_file_path)

# List all available KPIs along with their associated components and entity sets
kpi_summary = metrics_df[['entity_set', 'entity_name', 'metric']].drop_duplicates().sort_values(by=['entity_set', 'entity_name', 'metric'])

kpi_summary

```
Out[2]:
```
The `metrics.parquet` file contains various KPIs across multiple components and entity sets. Examples include `avg_request_latency_seconds` and `request_count` for APM instances like `accounting-cf54cf94b-cwql8`, as well as node-level metrics such as `node_disk_usage_rate`, `node_memory_usage_rate`, and `node_ready_status`. There are a total of 318 unique rows of KPIs, components, and entity sets available for analysis.

The original code execution output of IPython Kernel is also provided below for reference:

entity_set                 entity_name                       metric
78538  apm.instance  accounting-cf54cf94b-cwql8  avg_request_latency_seconds
71586  apm.instance  accounting-cf54cf94b-cwql8                request_count
74638  apm.instance  accounting-cf54cf94b-l667d  avg_request_latency_seconds
70716  apm.instance  accounting-cf54cf94b-l667d                request_count
74783  apm.instance  accounting-cf54cf94b-qtq2n  avg_request_latency_seconds
...             ...                         ...                          ...
1885       k8s.node                                     node_disk_usage_rate
4640       k8s.node                                   node_memory_usage_rate
1450       k8s.node                                    node_not_ready_status
4785       k8s.node                                   node_pod_running_count
0          k8s.node                                        node_ready_status

[318 rows x 3 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[3]:

# Ensure 'time' and 'value' columns are numeric for calculations
metrics_df['time'] = pd.to_numeric(metrics_df['time'], errors='coerce')
metrics_df['value'] = pd.to_numeric(metrics_df['value'], errors='coerce')

# Create a new column to represent 'component-KPI'
metrics_df['component_kpi'] = metrics_df['entity_name'] + '-' + metrics_df['metric']

# Group by 'component_kpi' and calculate the P95 threshold for each time series
p95_thresholds = metrics_df.groupby('component_kpi')['value'].quantile(0.95).reset_index()
p95_thresholds.columns = ['component_kpi', 'p95_threshold']

p95_thresholds

```
Out[3]:
```
The global P95 thresholds for each 'component-KPI' time series have been calculated using the entire `metrics.parquet` file. There are 318 unique 'component-KPI' combinations, each with its corresponding P95 threshold. Examples include `apiserver_concurrent_connections` with a threshold of 6.0 and `shipping::oteldemo.ShippingService/GetQuote-request_count` with a threshold of 18365.0. These thresholds can be used to identify anomalies in the metrics.

The original code execution output of IPython Kernel is also provided below for reference:

component_kpi  p95_threshold
0                    -apiserver_concurrent_connections       6.000000
1                  -apiserver_non_2xx_read_request_qps       0.088438
2                 -apiserver_non_2xx_write_request_qps       0.445084
3                       -apiserver_requests_per_second      27.079595
4                           -cluster_failed_node_count       0.000000
..                                                 ...            ...
313  shipping::oteldemo.ShippingService/GetQuote-av...       0.005571
314  shipping::oteldemo.ShippingService/GetQuote-re...   18365.000000
315  shipping::oteldemo.ShippingService/GetQuote-sl...       1.000000
316  shipping::oteldemo.ShippingService/ShipOrder-a...       0.000010
317  shipping::oteldemo.ShippingService/ShipOrder-r...   18365.000000

[318 rows x 2 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[4]:

import pytz
from datetime import datetime

# Define the alert time window in UTC+8
alert_start = datetime.fromisoformat('2026-04-25T05:18:12.716735+00:00').astimezone(pytz.timezone('Asia/Shanghai'))
alert_end = datetime.fromisoformat('2026-04-25T05:28:12.716735+00:00').astimezone(pytz.timezone('Asia/Shanghai'))

# Convert the alert time window to microseconds for comparison with metrics data
alert_start_us = int(alert_start.timestamp() * 1_000_000)
alert_end_us = int(alert_end.timestamp() * 1_000_000)

# Filter metrics data within the alert time window
filtered_metrics = metrics_df[(metrics_df['time'] >= alert_start_us) & (metrics_df['time'] <= alert_end_us)]

# Merge the filtered metrics with the P95 thresholds
anomalies = filtered_metrics.merge(p95_thresholds, on='component_kpi', how='left')

# Identify anomalies where the value exceeds the P95 threshold
anomalies = anomalies[anomalies['value'] > anomalies['p95_threshold']]

anomalies

```
Out[4]:
```
The filtered metrics data within the alert time window identified 595 anomalies where the metric values exceeded their respective global P95 thresholds. These anomalies span various components and KPIs, such as `node_memory_usage_rate` for Kubernetes nodes and `error_rate` for APM instances like `frontend-proxy` and `payment`. These anomalies indicate potential issues in these components during the alert period.

The original code execution output of IPython Kernel is also provided below for reference:

time domain    entity_set                         entity_id                      entity_name  ...      value                                     metric_set_id service                               component_kpi p95_threshold
1060   1777094322000000    k8s      k8s.node                                                                     ...  94.181222  k8s@metric_set@k8s.metric.high_level_metric_node    None                     -node_memory_usage_rate     93.079072
1061   1777094352000000    k8s      k8s.node                                                                     ...  94.467501  k8s@metric_set@k8s.metric.high_level_metric_node    None                     -node_memory_usage_rate     93.079072
1062   1777094382000000    k8s      k8s.node                                                                     ...  94.518452  k8s@metric_set@k8s.metric.high_level_metric_node    None                     -node_memory_usage_rate     93.079072
1063   1777094412000000    k8s      k8s.node                                                                     ...  94.643908  k8s@metric_set@k8s.metric.high_level_metric_node    None                     -node_memory_usage_rate     93.079072
1064   1777094442000000    k8s      k8s.node                                                                     ...  94.665304  k8s@metric_set@k8s.metric.high_level_metric_node    None                     -node_memory_usage_rate     93.079072
...                 ...    ...           ...                               ...                              ...  ...        ...                                               ...     ...                                         ...           ...
13153  1777094892000000    apm  apm.instance  d1f1b0349fadcfdc23ca3918ac448623  frontend-proxy-67d64dc4f8-xz2fd  ...   0.122761            apm@metric_set@apm.metric.apm.instance    None  frontend-proxy-67d64dc4f8-xz2fd-error_rate      0.122511
13167  1777094712000000    apm  apm.instance  8161f0b2fe6bbf0d49bcadedf8f40935         payment-696d688896-d2z22  ...   0.511371            apm@metric_set@apm.metric.apm.instance    None         payment-696d688896-d2z22-error_rate      0.509273
13173  1777094892000000    apm  apm.instance  8161f0b2fe6bbf0d49bcadedf8f40935         payment-696d688896-d2z22  ...   0.509609            apm@metric_set@apm.metric.apm.instance    None         payment-696d688896-d2z22-error_rate      0.509273
13183  1777094592000000    apm  apm.instance  4dab800a1af4e1c5967569ece76ece14         payment-696d688896-klnbs  ...   0.505198            apm@metric_set@apm.metric.apm.instance    None         payment-696d688896-klnbs-error_rate      0.505019
13190  1777094802000000    apm  apm.instance  4dab800a1af4e1c5967569ece76ece14         payment-696d688896-klnbs  ...   0.506027            apm@metric_set@apm.metric.apm.instance    None         payment-696d688896-klnbs-error_rate      0.505019

[595 rows x 11 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[5]:

# Sort anomalies by component_kpi and time for consecutive analysis
anomalies = anomalies.sort_values(by=['component_kpi', 'time'])

# Identify consecutive anomalies by calculating time differences
anomalies['time_diff'] = anomalies.groupby('component_kpi')['time'].diff().fillna(0)

# Define a threshold for consecutive anomalies (e.g., anomalies within 1 minute are considered consecutive)
consecutive_threshold_us = 60 * 1_000_000  # 1 minute in microseconds

# Mark consecutive anomalies
anomalies['is_consecutive'] = anomalies['time_diff'] <= consecutive_threshold_us

# Group anomalies by component_kpi and consecutive blocks
anomalies['consecutive_block'] = (anomalies['is_consecutive'] == False).cumsum()

# Filter out isolated noise spikes by keeping only blocks with more than one anomaly
faults = anomalies.groupby(['component_kpi', 'consecutive_block']).filter(lambda x: len(x) > 1)

# Extract relevant columns for fault analysis
faults_summary = faults[['component_kpi', 'time', 'value', 'p95_threshold']]

faults_summary

```
Out[5]:
```
The analysis identified 507 faults across various 'component-KPI' time series, where consecutive anomalies were detected, indicating persistent issues rather than isolated noise spikes. Examples include `deployment_cpu_usage_total` exceeding its P95 threshold consistently over multiple time points and `shipping::oteldemo.ShippingService/ShipOrder-avg_request_latency_seconds` showing repeated anomalies. These faults highlight components and KPIs that experienced sustained issues during the alert time window.

The original code execution output of IPython Kernel is also provided below for reference:

component_kpi              time     value  p95_threshold
7420                        -deployment_cpu_usage_total  1777094322000000  2.274393       1.254918
7421                        -deployment_cpu_usage_total  1777094352000000  2.264500       1.254918
7422                        -deployment_cpu_usage_total  1777094382000000  2.312771       1.254918
7423                        -deployment_cpu_usage_total  1777094412000000  2.479715       1.254918
7324                        -deployment_cpu_usage_total  1777094442000000  1.360429       1.254918
...                                                 ...               ...       ...            ...
9175  shipping::oteldemo.ShippingService/ShipOrder-a...  1777094652000000  0.000010       0.000010
9176  shipping::oteldemo.ShippingService/ShipOrder-a...  1777094682000000  0.000010       0.000010
9177  shipping::oteldemo.ShippingService/ShipOrder-a...  1777094712000000  0.000010       0.000010
9182  shipping::oteldemo.ShippingService/ShipOrder-a...  1777094862000000  0.000010       0.000010
9183  shipping::oteldemo.ShippingService/ShipOrder-a...  1777094892000000  0.000010       0.000010

[507 rows x 4 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[6]:

# Load the traces.parquet file
traces_file_path = 'rca100/cases/t001/traces.parquet'
traces_df = pd.read_parquet(traces_file_path)

# Ensure 'startTime' and 'endTime' are numeric for filtering
traces_df['startTime'] = pd.to_numeric(traces_df['startTime'], errors='coerce')
traces_df['endTime'] = pd.to_numeric(traces_df['endTime'], errors='coerce')

# Convert alert time window to nanoseconds for comparison with traces data
alert_start_ns = alert_start_us * 1_000  # Convert microseconds to nanoseconds
alert_end_ns = alert_end_us * 1_000  # Convert microseconds to nanoseconds

# Filter traces within the alert time window
filtered_traces = traces_df[
    (traces_df['startTime'] >= alert_start_ns) & (traces_df['endTime'] <= alert_end_ns)
]

# Identify traces related to the 'checkout::/oteldemo.CheckoutService/PlaceOrder' operation
checkout_traces = filtered_traces[
    filtered_traces['spanName'] == 'checkout::/oteldemo.CheckoutService/PlaceOrder'
]

# Analyze the call chain to determine the most downstream faulty service
# Extract relevant columns for analysis
call_chain_analysis = checkout_traces[['traceId', 'spanId', 'parentSpanId', 'serviceName', 'statusCode', 'duration']]

# Filter for faulty spans (statusCode != 0)
faulty_spans = call_chain_analysis[call_chain_analysis['statusCode'] != 0]

# Sort by traceId and duration to identify the most downstream faulty service
faulty_spans_sorted = faulty_spans.sort_values(by=['traceId', 'duration'], ascending=[True, False])

faulty_spans_sorted

```
Out[6]:
```
No traces related to the `checkout::/oteldemo.CheckoutService/PlaceOrder` operation within the alert time window were found to have a non-zero `statusCode`, indicating no faulty spans were detected in the call chain for this operation during the specified period.

The original code execution output of IPython Kernel is also provided below for reference:

Empty DataFrame
Columns: [traceId, spanId, parentSpanId, serviceName, statusCode, duration]
Index: []```
```

In [ ]:
# In[7]:

import re

# Escape faulty component names to avoid regex issues
escaped_faulty_components = [re.escape(component) for component in faulty_components]

# Search for error messages or anomalies related to the 'checkout' service or other faulty components
error_logs = filtered_logs[filtered_logs['content'].str.contains('|'.join(escaped_faulty_components), case=False, na=False)]

error_logs

```
Out[7]:
```
No error messages or anomalies related to the `checkout` service or other faulty components identified in the metrics analysis were found in the logs within the alert time window.

The original code execution output of IPython Kernel is also provided below for reference:

Empty DataFrame
Columns: [content, _time_, _source_, _container_ip_, _image_name_, _container_name_, _pod_name_, _namespace_, _pod_uid_, __topic__, __tag__:__pack_id__, __tag__:_cluster_id_, __tag__:__hostname__, __tag__:_node_name_, __tag__:_node_ip_]
Index: []

[0 rows x 15 columns]```
```

In [ ]:
# In[8]:

# Calculate the deviation from the P95 threshold for each fault
faults['deviation'] = faults['value'] - faults['p95_threshold']

# Identify the fault with the most significant deviation
most_significant_fault = faults.loc[faults['deviation'].idxmax()]

most_significant_fault

```
Out[8]:
```
The most significant fault identified is related to the `deployment_memory_usage_total` metric, where the memory usage exceeded its P95 threshold by approximately 3.65 GB. This indicates a memory pressure issue in a Kubernetes deployment, which is likely the root cause component and reason for the observed anomalies.

The original code execution output of IPython Kernel is also provided below for reference:

time                                                  1777094472000000
domain                                                             k8s
entity_set                                              k8s.deployment
entity_id                                                             
entity_name                                                           
metric                                   deployment_memory_usage_total
value                                                     5701558272.0
metric_set_id        k8s@metric_set@k8s.metric.high_level_metric_de...
service                                                           None
component_kpi                           -deployment_memory_usage_total
p95_threshold                                             2047787827.2
time_diff                                                   30000000.0
is_consecutive                                                    True
consecutive_block                                                    1
deviation                                                 3653770444.8
Name: 5205, dtype: object```
```